In [ ]:
Objetivo:

-Optimizar un modelo de IA en este caso RandomForest

In [2]:
#El codigo base que vamos a usar, es un codigo que tiene un modelo de Aprendizaje supervisado(RandomForest)
#Ire optimizandolo para mejorar sus caracteristicas.

import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

diabetes = datasets.load_diabetes()

diabetes_df = pd.DataFrame(data=np.c_[diabetes['data'], diabetes['target']],
                           columns=diabetes['feature_names'] + ['target'])

diabetes_df['target'] = (diabetes_df['target'] > diabetes_df['target'].median()).astype(int)

X = diabetes_df.drop('target', axis=1)
y = diabetes_df['target']
X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_entrena, y_entrena)

predicciones = modelo.predict(X_prueba)

puntaje = modelo.score(X_prueba, y_prueba)
print(f"\nPrecisión del modelo: {puntaje:.2f}")


Precisión del modelo: 0.72


In [ ]:
Optimizacion de codigo:

In [4]:
#Lo primero es importar las bibliotecas que vamos a usar:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline

In [5]:
# Aqui Cargamos el dataset de diabetes
diabetes = datasets.load_diabetes()

#Convertimos el dataset en un DataFrame, sera mucho mas sencillo ajustar los datos una vez convertido en DataFrame
diabetes_df = pd.DataFrame(data=np.c_[diabetes['data'], diabetes['target']],
                           columns=diabetes['feature_names'] + ['target'])

In [6]:
# Convertimos la variable 'target' en categórica para la clasificación
diabetes_df['target'] = (diabetes_df['target'] > diabetes_df['target'].median()).astype(int)

# Dividimos los datos en conjuntos de entrenamiento y prueba, utilizare una division de prueba al 20% y entrenamiento al 80%
#Tambien utilizo el random_state=42 para que se puedan comprobar mis resultados desde otros equipos
X = diabetes_df.drop('target', axis=1)
y = diabetes_df['target']
X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# Selecciono las 4 mejores características usando el test ANOVA F, uso esto para evitar perdida de informacion(data leakage)
selector = SelectKBest(score_func=f_classif, k=4)
X_entrena_seleccionada = selector.fit_transform(X_entrena, y_entrena)
X_prueba_seleccionada = selector.transform(X_prueba)

# Aqui creo el Pipeline sin la selección de características, usara 100 arboles de decision, tambien establezco el random_state=42 
#para que se puedan comprobar mis resultados desde otros equipos
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Entreno el modelo usando el pipeline anteriormente creado
pipeline.fit(X_entrena_seleccionada, y_entrena)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier', RandomForestClassifier(random_state=42))])

In [9]:
# Realizo las prediciones con el conjunto de prueba
predicciones = pipeline.predict(X_prueba_seleccionada)

# Evaluación del modelo
puntaje = pipeline.score(X_prueba_seleccionada, y_prueba)
print(f"Precisión del modelo: {puntaje:.2f}")

Precisión del modelo: 0.74


In [10]:
# A parte voy a hacer una evaluación adicional con validación cruzada, puesto que es mucho mas fiable que un solo split
puntajes_validacion_cruzada = cross_val_score(pipeline, selector.transform(X), y, cv=5)
print(f"Puntuaciones de validación cruzada: {puntajes_validacion_cruzada}")
print(f"Promedio de puntuación de validación cruzada: {np.mean(puntajes_validacion_cruzada):.2f}")

Puntuaciones de validación cruzada: [0.69662921 0.74157303 0.67045455 0.61363636 0.68181818]
Promedio de puntuación de validación cruzada: 0.68


In [ ]:
Conclusiones:

-Dataset: 442 muestras, 10 features médicas

-Problema: Clasificación binaria (por encima/debajo de la mediana)

-Precisión en prueba: 0.74

-Promedio validación cruzada: ~0.68

-posibles mejora con más datos o ajuste de hiperparámetros